In [ ]:
# RQ4: Which workout types are associated with the highest calorie expenditure?

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load dataset
df = pd.read_csv("gym_members_exercise_tracking_synthetic_data (2).csv")

# Clean Workout_Type column
df["Workout_Type"] = df["Workout_Type"].astype(str)
df["Workout_Type"] = df["Workout_Type"].str.replace(r"\\n|\\t|\n|\t", "", regex=True)
df["Workout_Type"] = df["Workout_Type"].str.strip()

# Remove missing workout type rows
df = df[df["Workout_Type"] != "nan"]

# Clean Calories_Burned column
df["Calories_Burned"] = pd.to_numeric(df["Calories_Burned"], errors="coerce")

# Drop rows with missing workout type or calories
df = df.dropna(subset=["Workout_Type", "Calories_Burned"])

# Check cleaned workout types
print("Workout Types:")
print(df["Workout_Type"].unique())

# Summary table
workout_summary = df.groupby("Workout_Type")["Calories_Burned"].agg(["mean", "median"]).reset_index()
workout_summary = workout_summary.sort_values(by="mean", ascending=False)

print("\nWorkout Summary:")
print(workout_summary)

# Chart style
sns.set_style("whitegrid")
colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]

# Bar chart
plt.figure(figsize=(8, 5))
sns.barplot(
    data=workout_summary,
    x="Workout_Type",
    y="mean",
    palette=colors
)

plt.title("Average Calories Burned by Workout Type", fontsize=14, weight="bold")
plt.xlabel("Workout Type")
plt.ylabel("Average Calories Burned")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# Boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="Workout_Type",
    y="Calories_Burned",
    order=workout_summary["Workout_Type"],
    palette=colors
)

plt.title("Calories Burned Distribution by Workout Type", fontsize=14, weight="bold")
plt.xlabel("Workout Type")
plt.ylabel("Calories Burned")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# ANOVA test
groups = []

for workout in workout_summary["Workout_Type"]:
    group = df[df["Workout_Type"] == workout]["Calories_Burned"]
    groups.append(group)

print("\nGroup Sizes:")
for workout, group in zip(workout_summary["Workout_Type"], groups):
    print(workout, ":", len(group))

f_stat, p_value = stats.f_oneway(*groups)

print("\nOne-way ANOVA Results")
print("F-statistic:", f_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Conclusion: The differences in calories burned between workout types are statistically significant.")
else:
    print("Conclusion: The differences in calories burned between workout types are not statistically significant.")